In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, confusion_matrix, classification_report
import json
from sklearn.linear_model import LogisticRegression


In [28]:

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)


In [29]:
# Load LSTM bits
with open('../../models/lstm_v2_meta.json', 'r') as f:
    meta = json.load(f)

In [30]:

class FocalLossBinary(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)
        alpha_t = torch.where(targets == 1, self.alpha, 1.0 - self.alpha)
        
        bce = nn.BCEWithLogitsLoss(reduction='none')(logits, targets)
        loss = alpha_t * (1 - pt) ** self.gamma * bce
        if self.reduction == 'mean': return loss.mean()
        return loss.sum()

class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)
    def forward(self, lstm_out):
        scores = self.attn(lstm_out)
        weights = torch.softmax(scores, dim=1)
        return (weights * lstm_out).sum(dim=1), weights.squeeze(-1)

class LeptoLSTM_v2(nn.Module):
    def __init__(self, sq_in, st_in, hid=128, ly=2, dp=0.3):
        super().__init__()
        self.conv = nn.Conv1d(in_channels=sq_in, out_channels=sq_in * 2, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.conv_dropout = nn.Dropout(dp)
        self.lstm = nn.LSTM(sq_in * 2, hid, ly, batch_first=True, bidirectional=True, dropout=dp)
        bi_dim = hid * 2
        self.attention = TemporalAttention(bi_dim)
        self.static_branch = nn.Sequential(nn.Linear(st_in, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(dp), nn.Linear(64, 32), nn.ReLU())
        self.head = nn.Sequential(nn.Linear(bi_dim + 32, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dp), nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dp/2), nn.Linear(64, 1))
    def forward(self, x_sq, x_st):
        x_sq_c = x_sq.transpose(1, 2)
        x_sq_c = self.conv(x_sq_c)
        x_sq_c = self.relu(x_sq_c)
        x_sq_c = self.conv_dropout(x_sq_c)
        x_sq_lstm = x_sq_c.transpose(1, 2)
        lo, _ = self.lstm(x_sq_lstm)
        ctx, _ = self.attention(lo)
        so = self.static_branch(x_st)
        return self.head(torch.cat([ctx, so], dim=1))


### Loading necessary Artifacts from previous models

In [31]:

model_l = LeptoLSTM_v2(meta['seq_input_dim'], meta['static_input_dim'])
model_l.load_state_dict(torch.load('../../models/lstm_v2_model.pth', map_location='cpu'))
model_l.eval()

sc_seq = joblib.load('../../models/lstm_v2_scaler_seq.pkl')
sc_stat = joblib.load('../../models/lstm_v2_scaler_stat.pkl')

# Load RF bits
model_r = joblib.load('../../models/best_model.pkl')
le = joblib.load('../../models/label_encoder.pkl')
drr_df = pd.read_csv('../../models/district_risk_rate.csv')
drr_map = dict(zip(drr_df.iloc[:,0], drr_df.iloc[:,1]))

# Prepare Aligned Data
test_df = pd.read_csv('../../data/processed/splitteddataset/lepto_monthly_test.csv', parse_dates=['YearMonth'])
test_df['District_enc'] = le.transform(test_df['District'])
test_df['District_risk_rate'] = test_df['District'].map(drr_map).astype(float)

In [32]:

def build_aligned(df, meta):
    X_sq, X_st, y_out, al_idx = [], [], [], []
    for d, g in df.groupby('District', observed=True):
        g = g.sort_values('YearMonth')
        idx = g.index.tolist()
        sq_v = g[meta['sequence_cols']].values.astype(np.float32)
        st_v = g[meta['static_cols']].values.astype(np.float32)
        lb = g['RiskLabel'].values
        for i in range(meta['seq_len'] - 1, len(g)):
            X_sq.append(sq_v[i - meta['seq_len'] + 1 : i + 1])
            X_st.append(st_v[i])
            y_out.append(lb[i])
            al_idx.append(idx[i])
    return np.array(X_sq), np.array(X_st), np.array(y_out), al_idx

X_q, X_s, y_t, idx_a = build_aligned(test_df, meta)


### Predictions for RF+ LSTM separately

In [33]:

# LSTM Prediction
X_q_s = sc_seq.transform(X_q.reshape(-1, X_q.shape[2])).reshape(X_q.shape)
X_s_s = sc_stat.transform(X_s)
with torch.no_grad():
    probs_l = torch.sigmoid(model_l(torch.tensor(X_q_s), torch.tensor(X_s_s))).flatten().numpy()

# RF Prediction
test_al = test_df.iloc[idx_a].copy()
X_rf = test_al.drop(columns=['District', 'RiskLabel', 'YearMonth'])

# Ensure X_rf features are exactly what model_r expects
if hasattr(model_r, 'feature_names_in_'):
    X_rf = X_rf[model_r.feature_names_in_]
probs_r = model_r.predict_proba(X_rf)[:, 1]

### Stacking Meta-Learner (Trained on full train set OOF(Out‑Of‑Fold)/predictions)

In [34]:
train_df = pd.read_csv('../../data/processed/splitteddataset/lepto_monthly_train.csv', parse_dates=['YearMonth'])
train_df['District_enc'] = le.fit_transform(train_df['District']) # fit_transform, though le is already fitted
train_df['District_risk_rate'] = train_df['District'].map(drr_map).astype(float)
X_q_tr, X_s_tr, y_tr, idx_a_tr = build_aligned(train_df, meta)

X_q_s_tr = sc_seq.transform(X_q_tr.reshape(-1, X_q_tr.shape[2])).reshape(X_q_tr.shape)
X_s_s_tr = sc_stat.transform(X_s_tr)
with torch.no_grad():
    probs_l_tr = torch.sigmoid(model_l(torch.tensor(X_q_s_tr), torch.tensor(X_s_s_tr))).flatten().numpy()

train_al = train_df.iloc[idx_a_tr].copy()
X_rf_tr = train_al.drop(columns=['District', 'RiskLabel', 'YearMonth'])
if hasattr(model_r, 'feature_names_in_'): X_rf_tr = X_rf_tr[model_r.feature_names_in_]
probs_r_tr = model_r.predict_proba(X_rf_tr)[:, 1]

meta_model = LogisticRegression(class_weight='balanced', C=0.05) # Strong regularization to avoid overfitting to train probs
meta_model.fit(np.c_[probs_l_tr, probs_r_tr], y_tr)
print(f'Meta-Learner Weights: LSTM={meta_model.coef_[0][0]:.4f}, RF={meta_model.coef_[0][1]:.4f}, Intercept={meta_model.intercept_[0]:.4f}')

# Ensemble Prediction
probs_e = meta_model.predict_proba(np.c_[probs_l, probs_r])[:, 1]
auc_e = roc_auc_score(y_t, probs_e)
print(f'Ensemble AUC: {auc_e:.4f}')

Meta-Learner Weights: LSTM=0.9122, RF=5.4602, Intercept=-3.1733
Ensemble AUC: 0.9071


### Summary

In [35]:

thresh = 0.33
def calc_metrics(probs, y):
    preds = (probs >= thresh).astype(int)
    tp = ((preds == 1) & (y == 1)).sum()
    tn = ((preds == 0) & (y == 0)).sum()
    fp = ((preds == 1) & (y == 0)).sum()
    fn = ((preds == 0) & (y == 1)).sum()
    sens = tp / (tp + fn + 1e-9)
    spec = tn / (tn + fp + 1e-9)
    prec = tp / (tp + fp + 1e-9)
    f1 = 2 * prec * sens / (prec + sens + 1e-9)
    return roc_auc_score(y, probs), average_precision_score(y, probs), sens, spec, f1

l_m = calc_metrics(probs_l, y_t)
r_m = calc_metrics(probs_r, y_t)
e_m = calc_metrics(probs_e, y_t)

res = pd.DataFrame({
    'Metric': ['ROC-AUC', 'AP', 'Sens', 'Spec', 'F1 HR'],
    'LSTM v2': l_m,
    'RF': r_m,
    'Ensemble': e_m
})
print('\nComparison Table:')
print(res)



Comparison Table:
    Metric   LSTM v2        RF  Ensemble
0  ROC-AUC  0.894978  0.905846  0.907147
1       AP  0.890118  0.913172  0.913409
2     Sens  0.986486  0.916667  0.880631
3     Spec  0.276507  0.602911  0.704782
4    F1 HR  0.712195  0.781190  0.800409
